# Parsing a CSV File with pandas

This is the same demo as `csv_parsing_demo.ipynb`, rebuilt with **pandas**. We work with `reviews.csv`, a file of video game user reviews:

1. Opening the CSV file
2. Checking the column names within the CSV file
3. Storing records and displaying information from the CSV file

> ⚠️ The `review_text` column contains real, unmoderated user reviews. Some are offensive, so check any text before you show it on screen.

In [5]:
import pandas as pd

FILE_PATH = "reviews.csv"

# Keep long review text from flooding the output
pd.set_option("display.max_colwidth", 60)

print(pd.__version__)

3.1.1


---
## Step 1: Opening the CSV file

With the standard `csv` module we had to open the file, build a reader and loop over the rows.

With pandas, **one call** does all of that. `pd.read_csv()` opens the file, parses it and closes it, and returns a **DataFrame**: a table with labelled columns and rows.

Like `csv.reader`, it understands quoted fields, so reviews containing commas or line breaks are parsed correctly.

In [6]:
df = pd.read_csv(FILE_PATH, encoding="utf-8")

type(df)

pandas.DataFrame

`.shape` gives `(rows, columns)`. pandas uses the header row for column names, so it isn't counted as a row.

The file has about 96,889 raw lines, but pandas correctly finds 46,105 records.

In [ ]:
rows, columns = df.shape
print(f"Rows: {rows:,}   Columns: {columns}")

`.head()` shows the first rows. In Jupyter, the last expression in a cell is shown as a formatted table.

In [ ]:
df.head(3)

**Big files:** you don't have to load everything at once. `nrows` reads only the first *n* records, which is handy for a quick look at a huge file.

In [ ]:
sample = pd.read_csv(FILE_PATH, nrows=5)
sample.shape

---
## Step 2: Checking the column names within the CSV file

The column names live in `df.columns`:

In [ ]:
df.columns

In [ ]:
for index, name in enumerate(df.columns, start=1):
    print(f"{index}. {name}")

Unlike the `csv` module, pandas **infers a type for each column**. `score` was read as a number (`int64`), not a string:

In [ ]:
df.dtypes

`.info()` shows the columns, their types, how many values are non-null and how much memory the table uses, all in one view:

In [ ]:
df.info()

Check that the columns the analysis needs actually exist:

In [ ]:
required_columns = ["platform", "score", "review_text"]

missing = [column for column in required_columns if column not in df.columns]
print("Missing columns:", missing if missing else "none")

---
## Step 3: Storing records and displaying information from the CSV file

### 3a. The records are already stored

The DataFrame *is* our storage. There's no need to append rows to a list ourselves.

To look at one record, use `.iloc[position]`. It returns a **Series**: one row, with the column names as labels.

In [ ]:
first_review = df.iloc[0]
first_review

In [ ]:
print(first_review["review_text"])

Select a column with `df["name"]`, or several columns with a list: `df[["a", "b"]]`.

In [ ]:
df[["platform", "published_date", "score"]].head()

### 3b. Clean the data

`score` is already an integer, but `published_date` was read as text. Convert it to a real date, and rename it to match the standard-library notebook:

In [ ]:
df["published_date"] = pd.to_datetime(df["published_date"])
df = df.rename(columns={"published_date": "date", "review_text": "text"})

df["text"] = df["text"].str.strip()

df.dtypes

### 3c. Display the first few records

Collapse each review onto one line and shorten it, so the output stays readable:

In [ ]:
preview = df[["platform", "date", "score", "text"]].head(3).copy()
preview["text"] = preview["text"].str.split().str.join(" ").str.slice(0, 70) + "..."

preview

### 3d. Answer questions about the data

Each loop from the standard-library version becomes a single method call.

**What is the average score?**

In [ ]:
print(f"Average score: {df['score'].mean():.2f} / 10")

`.describe()` gives the main statistics in one go:

In [ ]:
df["score"].describe()

**How many reviews per platform?** `.value_counts()` replaces the dictionary and the `Counter`:

In [ ]:
df["platform"].value_counts()

**What is the average score per platform?** `.groupby()` splits the table by platform and then aggregates each group:

In [ ]:
df.groupby("platform")["score"].agg(["count", "mean"]).round(2)

**How are the scores distributed?** A text bar chart:

In [ ]:
score_counts = df["score"].value_counts().sort_index()

for score, count in score_counts.items():
    bar = "█" * (count // 400)
    print(f"{score:>2} | {bar} {count:,}")

**How many reviews were published each day?**

In [ ]:
df["date"].dt.date.value_counts().sort_index()

**Filtering with a boolean mask:** `df["score"] == 10` gives a True/False value for every row. Indexing the DataFrame with it keeps only the matching rows.

In [ ]:
is_perfect = df["score"] == 10
is_perfect.head()

In [ ]:
perfect = df[is_perfect]
zero = df[df["score"] == 0]

print(f"Perfect 10s: {len(perfect):,} ({len(perfect) / len(df):.1%})")
print(f"Zeros:       {len(zero):,} ({len(zero) / len(df):.1%})")

Combine conditions with `&` (and) and `|` (or). Wrap each condition in parentheses:

In [ ]:
xbox_perfect = df[(df["platform"] == "xbox-series-x") & (df["score"] == 10)]
print(f"Perfect 10s on Xbox: {len(xbox_perfect):,}")

### 3e. A data-quality check: duplicate IDs

`review_id` *should* be unique. Let's check that:

In [ ]:
print(f"Rows:       {len(df):,}")
print(f"Unique IDs: {df['review_id'].nunique():,}")
print(f"Duplicates: {df['review_id'].duplicated().sum():,}")

`keep=False` marks *every* copy of a duplicated ID, so we can see the copies side by side:

In [ ]:
duplicates = df[df["review_id"].duplicated(keep=False)].sort_values("review_id")
duplicates[["review_id", "platform", "date", "score"]].head(6)

One way to clean them up is `drop_duplicates`:

In [ ]:
deduped = df.drop_duplicates(subset="review_id")
print(f"Before: {len(df):,}   After: {len(deduped):,}")

### 3f. Back to plain Python (optional)

If you need the standard-library shape of the data (a list of dicts), `to_dict("records")` converts it:

In [ ]:
records = df.head(2).to_dict("records")
records[0]["platform"], records[0]["score"]

---
## Recap: `csv` module vs pandas

| Task | Standard library | pandas |
|---|---|---|
| Open and parse | `with open(...)` + `csv.DictReader` | `pd.read_csv(path)` |
| Column names | `reader.fieldnames` | `df.columns` |
| Column types | Everything is a `str` | Inferred (`df.dtypes`) |
| Store records | Append to a list of dicts | The DataFrame itself |
| Average | Loop and sum | `df["score"].mean()` |
| Count values | `dict` / `Counter` | `.value_counts()` |
| Group and aggregate | Loop per group | `.groupby(...).agg(...)` |
| Filter | List comprehension | Boolean mask: `df[df["score"] == 10]` |
| Duplicates | `len(ids) - len(set(ids))` | `.duplicated().sum()` |

**When should you use which?** The `csv` module has no dependencies and processes one row at a time, which suits small scripts and very large streaming jobs. pandas is the better tool for exploring and analysing tabular data.

**Try it yourself:**
1. Find the longest review (hint: `df["text"].str.len()` and `.idxmax()`) and show its platform and score.
2. Count how many reviews on each platform scored 5 or less.
3. Save the perfect-10 reviews to `perfect_reviews.csv` using `.to_csv(..., index=False)`.